# dbstim walkthrough

Connect → choose an experiment → publish → download → present.

The server stores assignments, trial order and progress. Local files are only a media cache.
Run the cells one at a time.

## 1. Connect
`connect()` asks for your dbstimuli.com username and password; nothing is saved.

In [ ]:
from IPython.display import Video, display
import dbstim
from dbstim import MetricFilter

client = dbstim.connect()

## 2. Metrics you can filter on

In [ ]:
client.metrics()

## 3. Choose an experiment
List your experiments, then enter an exact name.

In [ ]:
client.experiments()

In [ ]:
experiment = client.experiment(input("Experiment name: "))
experiment.summary()

To **create one instead**, replace the preceding cell with:

```python
experiment = client.create_experiment(
    "Memory demo", subjects=2, runs_per_session=2,
    trials_per_run=12, foils_per_run=2, shared_fraction=0.1, seed=1,
    filters=[MetricFilter("duration_seconds", ">=", 10),
             MetricFilter("duration_seconds", "<=", 20)],
)
```

Each run then has eight full videos, two first halves and two second halves.
Use `client.experiment(id="revision-id")` if two experiments share a name.

## 4. Publish
Freeze the assignments and trial order. Publishing again changes nothing.

In [ ]:
experiment.publish()

## 5. Choose a subject and session
The server lists them; no ids to type.

In [ ]:
subject = experiment.subjects[0]
session = subject.sessions[0]
run = session.next_run()
run.trials if run else "Every trial is completed"

## 6. Download and preview
Downloading and previewing record no participant progress.

In [ ]:
session.download(f"downloads/{subject.id}/{session.id}")
trial = run.next_trial()
if trial is not None:
    display(Video(str(run.media(trial)), embed=True))

## 7. In your presentation program

`present(path)` is your PGL or PsychoPy function; it returns a JSON-compatible response.

```python
with session:  # starts the session; finishes it when the block ends
    while (run := session.next_run()) is not None:
        while (trial := run.next_trial()) is not None:
            path = run.media(trial)
            run.record(trial, "started")
            result = present(path)
            run.record(trial, "completed", result=result)
```

Leaving the block marks the session `completed` when every trial is completed,
`interrupted` if you stopped early, and `failed` if an error was raised.
Stop on errors; retry an unacknowledged call unchanged. A trial that started but never
completed needs review; it is never replayed automatically.

## 8. Check progress
The server keeps progress, so any device that opens this experiment sees the same state.

In [ ]:
session = experiment.subjects[0].sessions[0]
session.next_run()

In [ ]:
run.results